## **Data Sampling**

**Prerequisites:** Data Types, Statistics primer | **Next:** Cleaning & Missing Values | **Depth tier:** Foundation

## 1. Theory

A model can only be as good as the assumption that training data
represents the population it will be deployed on - sampling is where that
assumption gets made or broken.

## 2. The IID Assumption and Sampling Schemes

Most ML theory (including the generalization bound derived in
`07_ML_Theory\06_vc_dimension_and_pac.ipynb`) assumes training examples are

**independent and identically distributed (i.i.d.)** - drawn independently
from the same fixed distribution as future (test/deployment) data. This is
an assumption, not a guarantee - worth stating explicitly since violating
it (e.g. time-ordered data with drift, or data from a different population
than deployment) silently breaks the theoretical guarantees, even though
the algorithm will still run and produce output.

**Sampling schemes** (how you actually go from population to a dataset):
- **Simple random sampling**: every point equally likely to be chosen.
- **Stratified sampling**: sample within pre-defined subgroups
  (strata) proportionally - ensures rare subgroups aren't accidentally
  underrepresented; directly the mechanism behind `StratifiedKFold`
  already used conceptually in `05_Model_Evaluation\05_cross_validation.ipynb`.
- **Cluster sampling**: sample entire pre-existing groups (e.g. all
  records from a randomly chosen subset of hospitals) rather than
  individual points - cheaper to collect but correlated within a cluster,
  reducing effective sample size relative to the same $n$ drawn by simple
  random sampling.
- **Systematic sampling**: every $k$-th record - practical but risks
  hidden periodicity in the data aligning with the sampling interval.


**Sampling bias / selection bias**: the sample systematically differs from
the population (e.g. only surveying people who respond to surveys -
"survivorship bias" is a specific, famous case of this: analyzing only
surviving/successful cases, e.g. only currently-active customers, silently
excludes churned ones from a churn-prediction training set, which is a
real, common, easy-to-miss bug).

## 3. Worked Numerical Example

Population: 1000 patients, 950 without a rare disease, 50 with it.
Simple random sample of $n=100$: expected count with disease
$=100\times50/1000=5$ - small enough that a single unlucky draw could give
0 or 1 positive cases, badly limiting what a model can learn about the
minority class. Stratified sample of $n=100$ preserving the 95/5 ratio
exactly: guarantees 5 positive, 95 negative - same total $n$, far more
reliable representation of the minority class (still small in absolute
terms - this is exactly why `02_Data/10_imbalanced_data_handling.ipynb`'s
SMOTE/class-weighting techniques exist on top of stratified sampling, not
instead of it).

In [2]:
import numpy as np

rng = np.random.default_rng(0)
population = np.array([0]*950 + [1]*50)

def simple_random_positive_count(pop, n, trials=1000, seed=0):
    r = np.random.default_rng(seed)
    return [r.choice(pop, size=n, replace=False).sum() for _ in range(trials)]

counts = simple_random_positive_count(population, 100)
print(np.mean(counts), np.std(counts))   # mean ~5, but real spread (std) across draws
print(min(counts), max(counts))          # some draws get very few or zero positives

5.056 2.0743345921041767
0 14


## 4. Failure Cases
Training a model on data collected only from users who opted into data
collection (self-selection bias) and deploying it on the full user base -
a real, common production ML failure mode, not a textbook hypothetical.

## 5. Assumptions
i.i.d. (stated above) - genuinely violated in common real cases: time
series (temporal dependence - direct link to
`05_Model_Evaluation/07_validation_strategies.ipynb`'s time-series-split
content), grouped data (e.g. multiple rows per patient - direct link to
that same file's group-k-fold content).